# CFEU journal extension: Notebook 5, robustness extensions for reviewers

Run after Notebooks 1 to 3. It reads their outputs from `cfeu_journal` in your Google Drive.

**What it tests**
- **A. Hyperparameter tuning:** nested cross-validation. Tuning happens only inside each training fold, and the outer folds measure performance. Compares tuned models with the hand-set configuration used in the paper.
- **B. Overfitting on Berka (682 loans):**
  - nested tuning
  - spread across 20 cross-validation repeats
  - gap between training and test AUC
  - a label-permutation test
  - a learning curve
  - an out-of-time test (train on early loans, test on later ones)
  - 3-, 6- and 12-month transaction windows
- **C. Class imbalance:** no correction, class weights, focal loss, undersampling and SMOTE.
- **D. External scores:** the models re-run without the undocumented second external score, and without all three external scores.
- **E. Fairness mitigation:**
  - reweighing
  - removing features that act as proxies for age
  - group-specific approval thresholds (equal opportunity, and equal approval rates)

**Runtime:** about 75 to 100 minutes. Each section saves its result to Drive as soon as it finishes. If Colab disconnects, run all again and finished sections are skipped.

**Send back to Claude:** `robustness_results.json`, which downloads at the end. It holds summary numbers only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
OUT = '/content/drive/MyDrive/cfeu_journal'
PARTS = f'{OUT}/nb5_parts'; os.makedirs(PARTS, exist_ok=True)
!pip -q install lightgbm pyarrow

In [ ]:
# Shared functions (no need to edit)
# ---- Modelling and evaluation (Notebook 2) ----
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_LGB = False


def make_model(kind, seed, small_data=False):
    if kind == "logit":
        return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                             LogisticRegression(max_iter=2000, C=1.0))
    if HAVE_LGB:
        p = dict(n_estimators=150, learning_rate=0.05, num_leaves=7, min_child_samples=20,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8) if small_data else \
            dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
        return lgb.LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, **p)
    return HistGradientBoostingClassifier(max_iter=150 if small_data else 400, learning_rate=0.05,
                                          max_leaf_nodes=7 if small_data else 31, random_state=seed)


def oof_predictions(X, y, kind, seeds, n_splits=5, small_data=False):
    """Out-of-fold predicted default probabilities, one column per seed."""
    P = np.zeros((len(y), len(seeds)))
    for j, s in enumerate(seeds):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=s)
        for tr, te in skf.split(X, y):
            m = make_model(kind, s, small_data)
            m.fit(X.iloc[tr], y[tr])
            P[te, j] = m.predict_proba(X.iloc[te])[:, 1]
    return P


def ks_stat(y, p):
    return float(stats.ks_2samp(p[y == 1], p[y == 0]).statistic)


def approval_metrics(y, p, approve_rate=0.7):
    """Approve the lowest-risk share; report default rate among approved."""
    cut = np.quantile(p, approve_rate)
    ap = p <= cut
    return {"approve_rate": approve_rate, "default_rate_approved": float(y[ap].mean()),
            "default_rate_declined": float(y[~ap].mean()), "threshold": float(cut)}


def calib_bins(y, p, n_bins=10):
    q = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "bin": q}).groupby("bin").agg(pred=("p", "mean"), obs=("y", "mean"), n=("y", "size"))
    return d.reset_index().to_dict(orient="list")


def bootstrap_auc_ci(y, p, n_boot=500, seed=0):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        vals.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


# Fast DeLong test (Sun & Xu 2014) for two correlated AUCs on the same cases
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2


def delong_test(y, p1, p2):
    y = np.asarray(y); order = np.argsort(-y, kind="mergesort")
    y = y[order]; preds = np.vstack([p1, p2])[:, order]
    m = int(y.sum()); n = len(y) - m
    pos, neg = preds[:, :m], preds[:, m:]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    diff = aucs[0] - aucs[1]
    se = np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    z = diff / se if se > 0 else 0.0
    return {"auc_1": float(aucs[0]), "auc_2": float(aucs[1]), "diff": float(diff),
            "z": float(z), "p_value": float(2 * stats.norm.sf(abs(z)))}


def evaluate(y, P, mask=None, n_boot=500):
    """P: n x seeds out-of-fold probabilities. Seed spread plus bootstrap CI on the seed-averaged score."""
    y = np.asarray(y)
    if mask is not None:
        y, P = y[mask], P[mask]
    per_seed = [roc_auc_score(y, P[:, j]) for j in range(P.shape[1])]
    p = P.mean(axis=1)
    return {"n": int(len(y)), "default_rate": float(y.mean()),
            "auc_mean_over_seeds": float(np.mean(per_seed)), "auc_sd_over_seeds": float(np.std(per_seed)),
            "auc": float(roc_auc_score(y, p)), "auc_ci95": bootstrap_auc_ci(y, p, n_boot),
            "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, p)),
            "ks": ks_stat(y, p), "approval_at_70pct": approval_metrics(y, p, 0.7),
            "calibration": calib_bins(y, p)}


def run_suite(df, y, feature_sets, seeds, subgroups, small_data=False, n_boot=500, log=print):
    """Train every learner on every feature set; evaluate overall and per subgroup; DeLong vs the first set."""
    y = np.asarray(y).astype(int)
    subgroups = {g: (None if m is None else np.asarray(m, bool)) for g, m in subgroups.items()}
    results, oof = {}, {}
    for kind in ["logit", "gbm"]:
        for name, cols in feature_sets.items():
            log(f"  {kind:5s} | {name:12s} | {len(cols)} features")
            P = oof_predictions(df[cols], y, kind, seeds, small_data=small_data)
            oof[(kind, name)] = P
            results[f"{kind}|{name}"] = {g: evaluate(y, P, m, n_boot) for g, m in subgroups.items()}
    base = list(feature_sets)[0]
    tests = {}
    for kind in ["logit", "gbm"]:
        for name in list(feature_sets)[1:]:
            for g, m in subgroups.items():
                mm = np.ones(len(y), bool) if m is None else m
                tests[f"{kind}|{name} vs {base}|{g}"] = delong_test(
                    y[mm], oof[(kind, name)][mm].mean(1), oof[(kind, base)][mm].mean(1))
    return results, tests, oof


def headline_table(results):
    rows = []
    for key, groups in results.items():
        kind, fs = key.split("|")
        for g, r in groups.items():
            rows.append({"learner": kind, "features": fs, "group": g, "n": r["n"],
                         "AUC": round(r["auc"], 3), "AUC 95% CI": f"{r['auc_ci95'][0]:.3f}-{r['auc_ci95'][1]:.3f}",
                         "AUC sd (seeds)": round(r["auc_sd_over_seeds"], 4), "PR-AUC": round(r["pr_auc"], 3),
                         "Brier": round(r["brier"], 4), "KS": round(r["ks"], 3),
                         "Default rate if 70% approved": round(r["approval_at_70pct"]["default_rate_approved"], 4)})
    return pd.DataFrame(rows)


# ---- Home Credit feature building ----
import os, json
import numpy as np
import pandas as pd


def _safe_div(a, b):
    a = np.asarray(a, dtype="float64"); b = np.asarray(b, dtype="float64")
    out = np.full_like(a, np.nan)
    m = (b != 0) & ~np.isnan(b)
    out[m] = a[m] / b[m]
    return out


def build_home_credit(hc_dir):
    """Return one row per applicant with TARGET, feature groups and audit-only fields."""
    app = pd.read_csv(os.path.join(hc_dir, "application_train.csv"), usecols=[
        "SK_ID_CURR", "TARGET", "CODE_GENDER", "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
        "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE",
        "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"])
    df = app[["SK_ID_CURR", "TARGET"]].copy()

    # Audit-only fields: never used as model inputs
    df["audit_gender"] = app["CODE_GENDER"].where(app["CODE_GENDER"].isin(["M", "F"]))
    age = -app["DAYS_BIRTH"] / 365.25
    df["audit_age"] = age.round(1)
    df["audit_age_band"] = pd.cut(age, [0, 30, 50, 200], labels=["<30", "30-49", "50+"], right=False).astype(str)
    df["contract_type"] = app["NAME_CONTRACT_TYPE"]

    # Bureau group: external scores + credit bureau records
    for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
        df["bur_" + c.lower()] = app[c]

    bur = pd.read_csv(os.path.join(hc_dir, "bureau.csv"), usecols=[
        "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE",
        "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CNT_CREDIT_PROLONG"])
    bur["is_active"] = (bur["CREDIT_ACTIVE"] == "Active").astype(int)
    g = bur.groupby("SK_ID_CURR")
    b = pd.DataFrame({
        "bur_n_loans": g.size(),
        "bur_n_active": g["is_active"].sum(),
        "bur_oldest_years": -g["DAYS_CREDIT"].min() / 365.25,
        "bur_newest_years": -g["DAYS_CREDIT"].max() / 365.25,
        "bur_max_days_overdue": g["CREDIT_DAY_OVERDUE"].max(),
        "bur_total_credit": g["AMT_CREDIT_SUM"].sum(),
        "bur_total_debt": g["AMT_CREDIT_SUM_DEBT"].sum(),
        "bur_total_overdue": g["AMT_CREDIT_SUM_OVERDUE"].sum(),
        "bur_n_prolonged": g["CNT_CREDIT_PROLONG"].sum(),
    })
    b["bur_debt_to_credit"] = _safe_div(b["bur_total_debt"], b["bur_total_credit"])

    bb_path = os.path.join(hc_dir, "bureau_balance.csv")
    if os.path.exists(bb_path):
        bb = pd.read_csv(bb_path, usecols=["SK_ID_BUREAU", "STATUS"])
        bb["late"] = bb["STATUS"].astype(str).isin(list("12345")).astype(int)
        bb = bb.groupby("SK_ID_BUREAU")["late"].agg(["sum", "count"]).reset_index()
        bb = bb.merge(bur[["SK_ID_BUREAU", "SK_ID_CURR"]], on="SK_ID_BUREAU")
        bb = bb.groupby("SK_ID_CURR")[["sum", "count"]].sum()
        b["bur_late_month_share"] = _safe_div(bb["sum"].reindex(b.index), bb["count"].reindex(b.index))
    df = df.merge(b, left_on="SK_ID_CURR", right_index=True, how="left")
    df["thin_file"] = df["bur_n_loans"].isna().astype(int)
    df["bur_n_loans"] = df["bur_n_loans"].fillna(0)
    df["bur_n_active"] = df["bur_n_active"].fillna(0)

    # Alternative group: affordability from the application + repayment behaviour outside the bureau
    df["alt_income"] = app["AMT_INCOME_TOTAL"]
    df["alt_credit_to_income"] = _safe_div(app["AMT_CREDIT"], app["AMT_INCOME_TOTAL"])
    df["alt_annuity_to_income"] = _safe_div(app["AMT_ANNUITY"], app["AMT_INCOME_TOTAL"])
    df["alt_credit_to_goods"] = _safe_div(app["AMT_CREDIT"], app["AMT_GOODS_PRICE"])
    emp = app["DAYS_EMPLOYED"].where(app["DAYS_EMPLOYED"] != 365243)  # 365243 is a placeholder for not employed
    df["alt_employment_years"] = -emp / 365.25
    df["alt_not_employed"] = (app["DAYS_EMPLOYED"] == 365243).astype(int)
    df["alt_income_type"] = app["NAME_INCOME_TYPE"].astype("category").cat.codes

    prev = pd.read_csv(os.path.join(hc_dir, "previous_application.csv"),
                       usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS", "AMT_APPLICATION", "AMT_CREDIT"])
    prev["approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
    prev["refused"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
    gp = prev.groupby("SK_ID_CURR")
    p = pd.DataFrame({"alt_prev_n": gp.size(), "alt_prev_approved_share": gp["approved"].mean(),
                      "alt_prev_refused_share": gp["refused"].mean()})

    ins = pd.read_csv(os.path.join(hc_dir, "installments_payments.csv"), usecols=[
        "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"],
        dtype={"SK_ID_CURR": "int32", "DAYS_INSTALMENT": "float32", "DAYS_ENTRY_PAYMENT": "float32",
               "AMT_INSTALMENT": "float32", "AMT_PAYMENT": "float32"})
    ins["days_late"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["is_late"] = (ins["days_late"] > 0).astype("int8")
    ins["is_short"] = (ins["AMT_PAYMENT"] < ins["AMT_INSTALMENT"] * 0.99).astype("int8")
    gi = ins.groupby("SK_ID_CURR")
    i = pd.DataFrame({"alt_inst_n": gi.size(), "alt_inst_late_share": gi["is_late"].mean(),
                      "alt_inst_mean_days_late": gi["days_late"].mean(), "alt_inst_max_days_late": gi["days_late"].max(),
                      "alt_inst_short_share": gi["is_short"].mean(),
                      "alt_inst_paid_ratio": gi["AMT_PAYMENT"].sum() / gi["AMT_INSTALMENT"].sum().replace(0, np.nan)})
    del ins

    pos = pd.read_csv(os.path.join(hc_dir, "POS_CASH_balance.csv"), usecols=["SK_ID_CURR", "SK_DPD_DEF"])
    pos["dpd"] = (pos["SK_DPD_DEF"] > 0).astype("int8")
    gq = pos.groupby("SK_ID_CURR")
    q = pd.DataFrame({"alt_pos_dpd_share": gq["dpd"].mean(), "alt_pos_max_dpd": gq["SK_DPD_DEF"].max()})
    del pos

    cc = pd.read_csv(os.path.join(hc_dir, "credit_card_balance.csv"),
                     usecols=["SK_ID_CURR", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "SK_DPD_DEF"])
    cc["util"] = _safe_div(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["dpd"] = (cc["SK_DPD_DEF"] > 0).astype("int8")
    gc = cc.groupby("SK_ID_CURR")
    c = pd.DataFrame({"alt_cc_util_mean": gc["util"].mean(), "alt_cc_dpd_share": gc["dpd"].mean()})
    del cc

    for t in (p, i, q, c):
        df = df.merge(t, left_on="SK_ID_CURR", right_index=True, how="left")
    df["alt_prev_n"] = df["alt_prev_n"].fillna(0)
    df["alt_inst_n"] = df["alt_inst_n"].fillna(0)
    return df


def home_credit_summary(df):
    bur_cols = [c for c in df.columns if c.startswith("bur_")]
    alt_cols = [c for c in df.columns if c.startswith("alt_")]
    out = {"n_applicants": int(len(df)), "default_rate": round(float(df["TARGET"].mean()), 4),
           "n_bureau_features": len(bur_cols), "n_alternative_features": len(alt_cols), "groups": {}}
    for name, sub in [("thin_file", df[df.thin_file == 1]), ("with_bureau", df[df.thin_file == 0])]:
        out["groups"][name] = {
            "n": int(len(sub)), "share": round(len(sub) / len(df), 4),
            "default_rate": round(float(sub["TARGET"].mean()), 4),
            "ext_source_missing_share": {c: round(float(sub[c].isna().mean()), 4)
                                         for c in ["bur_ext_source_1", "bur_ext_source_2", "bur_ext_source_3"]},
            "has_previous_loans_share": round(float((sub["alt_prev_n"] > 0).mean()), 4),
            "has_instalment_history_share": round(float((sub["alt_inst_n"] > 0).mean()), 4),
        }
    out["by_gender"] = df.groupby("audit_gender")["TARGET"].agg(["size", "mean"]).round(4).to_dict()
    out["by_age_band"] = df.groupby("audit_age_band")["TARGET"].agg(["size", "mean"]).round(4).to_dict()
    out["missing_share_top10"] = (df[bur_cols + alt_cols].isna().mean().sort_values(ascending=False)
                                  .head(10).round(4).to_dict())
    return out


# ---- Berka (PKDD'99) cash-flow features ----
CREDIT_TYPES = {"PRIJEM", "CREDIT"}
WITHDRAWAL_TYPES = {"VYDAJ", "VYBER", "WITHDRAWAL", "WITHDRAWAL IN CASH"}
SANCTION_SYMBOLS = {"SANKC. UROK", "SANCTION INTEREST IF NEGATIVE BALANCE"}


def _to_date(s):
    """Accept DATE values or YYMMDD integers from the original files."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    st = s.astype(str).str.replace(r"\.0$", "", regex=True)
    if st.str.fullmatch(r"\d{6}").all():
        return pd.to_datetime("19" + st, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(st, errors="coerce")


def _gender_birth_from_number(bn):
    """Birth number YYMMDD; women have 50 added to the month."""
    st = bn.astype(str).str.zfill(6)
    mm = st.str[2:4].astype(int)
    gender = np.where(mm > 50, "F", "M")
    mm = np.where(mm > 50, mm - 50, mm)
    birth = pd.to_datetime("19" + st.str[:2] + pd.Series(mm, index=st.index).astype(str).str.zfill(2) + st.str[4:6],
                           format="%Y%m%d", errors="coerce")
    return pd.Series(gender, index=bn.index), birth


def build_berka(tables, window_days=365, min_history_days=90, gap_days=0):
    """tables: dict of DataFrames with keys loan, account, trans, disp, client. One row per loan."""
    loan = tables["loan"].copy(); acc = tables["account"].copy(); tr = tables["trans"].copy()
    disp = tables["disp"].copy(); cl = tables["client"].copy()
    for t in (loan, acc, tr, disp, cl):
        t.columns = [c.lower() for c in t.columns]
    loan["date"] = _to_date(loan["date"]); acc["date"] = _to_date(acc["date"]); tr["date"] = _to_date(tr["date"])
    tr["type"] = tr["type"].astype(str).str.upper().str.strip()
    tr["k_symbol"] = tr["k_symbol"].astype(str).str.upper().str.strip()

    loan["default"] = loan["status"].astype(str).str.upper().isin(["B", "D"]).astype(int)
    loan = loan.merge(acc[["account_id", "date"]].rename(columns={"date": "acct_open"}), on="account_id", how="left")

    owner = disp[disp["type"].astype(str).str.upper() == "OWNER"][["account_id", "client_id"]]
    if "birth_number" in cl.columns:
        cl["audit_gender"], cl["birth"] = _gender_birth_from_number(cl["birth_number"])
    else:  # pre-processed versions carry gender and birth_date directly
        cl["audit_gender"] = cl["gender"].astype(str).str.upper().str[0]
        cl["birth"] = pd.to_datetime(cl["birth_date"], errors="coerce")
    owner = owner.merge(cl[["client_id", "audit_gender", "birth"]], on="client_id", how="left").drop_duplicates("account_id")
    loan = loan.merge(owner[["account_id", "audit_gender", "birth"]], on="account_id", how="left")
    loan["audit_age"] = ((loan["date"] - loan["birth"]).dt.days / 365.25).round(1)

    tr = tr.merge(loan[["account_id", "date"]].rename(columns={"date": "loan_date"}), on="account_id")
    end = tr["loan_date"] - pd.Timedelta(days=gap_days)  # gap_days > 0 leaves a buffer before the loan date
    tr = tr[(tr["date"] < end) & (tr["date"] >= end - pd.Timedelta(days=window_days))]
    tr["is_credit"] = tr["type"].isin(CREDIT_TYPES)
    tr["inflow"] = np.where(tr["is_credit"], tr["amount"].abs(), 0.0)
    tr["outflow"] = np.where(~tr["is_credit"], tr["amount"].abs(), 0.0)
    tr["cash_out"] = np.where(tr["operation"].astype(str).str.upper().str.contains("VYBER|CASH"), tr["amount"].abs(), 0.0)
    tr["sanction"] = tr["k_symbol"].isin(SANCTION_SYMBOLS).astype(int)
    tr["month"] = tr["date"].dt.to_period("M")

    m = tr.groupby(["account_id", "month"]).agg(inflow=("inflow", "sum"), outflow=("outflow", "sum"),
                                                min_bal=("balance", "min")).reset_index()
    gm = m.groupby("account_id")
    feats = pd.DataFrame({
        "cf_months_observed": gm.size(),
        "cf_inflow_monthly_mean": gm["inflow"].mean(),
        "cf_inflow_cv": gm["inflow"].std() / gm["inflow"].mean().replace(0, np.nan),
        "cf_expense_to_income": gm["outflow"].sum() / gm["inflow"].sum().replace(0, np.nan),
        "cf_months_net_negative": (m.assign(neg=m["outflow"] > m["inflow"]).groupby("account_id")["neg"].mean()),
        "cf_min_balance": gm["min_bal"].min(),
    })
    gt = tr.groupby("account_id")
    feats["cf_balance_mean"] = gt["balance"].mean()
    feats["cf_balance_volatility"] = gt["balance"].std() / gt["balance"].mean().abs().replace(0, np.nan)
    feats["cf_low_balance_share"] = gt["balance"].apply(lambda b: float((b < 5000).mean()))
    feats["cf_negative_balance_share"] = gt["balance"].apply(lambda b: float((b < 0).mean()))
    feats["cf_sanction_count"] = gt["sanction"].sum()
    feats["cf_cash_withdrawal_share"] = gt["cash_out"].sum() / gt["outflow"].sum().replace(0, np.nan)
    feats["cf_txn_per_month"] = gt.size() / feats["cf_months_observed"]
    first = gt["date"].min()

    out = loan.merge(feats, left_on="account_id", right_index=True, how="left")
    out = out.merge(first.rename("first_txn"), left_on="account_id", right_index=True, how="left")
    out["cf_history_days"] = (out["date"] - out["first_txn"]).dt.days
    out["cf_account_tenure_years"] = (out["date"] - out["acct_open"]).dt.days / 365.25
    out["terms_amount"] = out["amount"]; out["terms_duration"] = out["duration"]; out["terms_payment"] = out["payments"]
    out["cf_payment_to_income"] = out["payments"] / out["cf_inflow_monthly_mean"].replace(0, np.nan)
    out["enough_history"] = (out["cf_history_days"] >= min_history_days).astype(int)
    keep = (["loan_id", "account_id", "date", "status", "default", "enough_history", "audit_gender", "audit_age"]
            + [c for c in out.columns if c.startswith("cf_") or c.startswith("terms_")])
    return out[keep].rename(columns={"date": "loan_date"})


def berka_summary(df):
    cf = [c for c in df.columns if c.startswith("cf_")]
    return {"n_loans": int(len(df)), "n_defaults": int(df["default"].sum()),
            "default_rate": round(float(df["default"].mean()), 4),
            "status_counts": df["status"].astype(str).value_counts().to_dict(),
            "loans_with_90d_history": int(df["enough_history"].sum()),
            "n_cashflow_features": len(cf),
            "by_gender": df.groupby("audit_gender")["default"].agg(["size", "mean"]).round(4).to_dict(),
            "cashflow_feature_means_by_outcome": df.groupby("default")[cf].mean().round(3).to_dict()}


# ---- Robustness extensions (Notebook 5): tuning, overfitting, class imbalance, external scores, fairness mitigation ----
import time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold, ParameterSampler, ParameterGrid, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

HC_FIXED = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
BK_FIXED = dict(n_estimators=150, learning_rate=0.05, num_leaves=7, min_child_samples=20,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8)

# Search spaces. Home Credit: random search with early stopping picks the number of trees.
HC_SPACE = {"num_leaves": [7, 15, 31, 63, 127], "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
            "min_child_samples": [20, 50, 100, 200, 400], "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0], "reg_lambda": [0.0, 0.1, 1.0, 5.0, 10.0]}
# Berka: full grid, number of trees searched directly (early stopping on ~110 validation loans is too noisy).
BK_GRID = {"num_leaves": [3, 7, 15], "n_estimators": [50, 150, 300], "min_child_samples": [10, 20, 40]}
LOGIT_C = [0.001, 0.01, 0.1, 1.0, 10.0]


def lgbm(params, seed, **extra):
    p = {"subsample_freq": 1, **params, **extra}
    return lgb.LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, **p)


def logit(C=1.0):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(max_iter=2000, C=C))


def _sub(X, y, max_rows, seed):
    if max_rows is None or len(y) <= max_rows:
        return X, y
    Xs, _, ys, _ = train_test_split(X, y, train_size=max_rows, stratify=y, random_state=seed)
    return Xs, np.asarray(ys)


# ---------- metrics ----------
def ece(y, p, n_bins=10):
    """Expected calibration error over deciles of predicted risk."""
    q = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "b": q}).groupby("b").agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((d["n"] * (d["p"] - d["y"]).abs()).sum() / d["n"].sum())


def auc_ci(y, p, n_boot=200, seed=0):
    rng = np.random.default_rng(seed); n = len(y); v = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() != y[i].max():
            v.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]


def decision_metrics(y, p, approve_rate=0.7):
    cut = np.quantile(p, approve_rate); ap = p <= cut; dec = ~ap
    return {"default_rate_approved": float(y[ap].mean()),
            "share_of_defaulters_declined": float(dec[y == 1].mean()),       # recall on defaults
            "default_rate_among_declined": float(y[dec].mean()),              # precision on defaults
            "approval_rate_good_borrowers": float(ap[y == 0].mean())}


def summarise(y, p, mask=None, n_boot=200):
    y = np.asarray(y); p = np.asarray(p)
    if mask is not None:
        y, p = y[mask], p[mask]
    return {"n": int(len(y)), "auc": float(roc_auc_score(y, p)), "auc_ci95": auc_ci(y, p, n_boot),
            "pr_auc": float(average_precision_score(y, p)), "brier": float(brier_score_loss(y, np.clip(p, 0, 1))),
            "ece": ece(y, p), **decision_metrics(y, p)}


# ---------- A. hyperparameter tuning inside nested cross-validation ----------
def tune_lgbm(X, y, seed, space=None, n_iter=12, inner_folds=3, max_rows=50000, early_stop=True, base=None):
    """Random (or full-grid) search on the training data only. Returns best params and the search trace."""
    Xs, ys = _sub(X, y, max_rows, seed)
    if early_stop:
        cands = list(ParameterSampler(space, n_iter, random_state=seed))
        fixed = {k: v for k, v in (base or HC_FIXED).items() if k in space}
        cands.insert(0, fixed)  # the hand-set configuration always competes
    else:
        cands = list(ParameterGrid(space))
    skf = StratifiedKFold(inner_folds, shuffle=True, random_state=seed)
    splits = list(skf.split(Xs, ys)); trace = []
    for c in cands:
        aucs, iters = [], []
        for tr, va in splits:
            if early_stop:
                m = lgbm({**(base or HC_FIXED), **c, "n_estimators": 2000}, seed, metric="auc")
                m.fit(Xs.iloc[tr], ys[tr], eval_set=[(Xs.iloc[va], ys[va])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False)])
                iters.append(m.best_iteration_ or 2000)
            else:
                m = lgbm({**(base or BK_FIXED), **c}, seed).fit(Xs.iloc[tr], ys[tr])
            aucs.append(roc_auc_score(ys[va], m.predict_proba(Xs.iloc[va])[:, 1]))
        rec = {**c, "inner_auc": float(np.mean(aucs))}
        if early_stop:
            rec["n_estimators"] = int(max(50, np.mean(iters) * 1.1))
        trace.append(rec)
    best = max(trace, key=lambda r: r["inner_auc"])
    params = {**(base or (HC_FIXED if early_stop else BK_FIXED)), **{k: v for k, v in best.items() if k != "inner_auc"}}
    return params, trace


def tune_logit(X, y, seed, inner_folds=3, max_rows=50000):
    Xs, ys = _sub(X, y, max_rows, seed)
    skf = StratifiedKFold(inner_folds, shuffle=True, random_state=seed); res = {}
    for C in LOGIT_C:
        a = [roc_auc_score(ys[va], logit(C).fit(Xs.iloc[tr], ys[tr]).predict_proba(Xs.iloc[va])[:, 1])
             for tr, va in skf.split(Xs, ys)]
        res[C] = float(np.mean(a))
    return max(res, key=res.get), res


def nested_cv(X, y, kind, outer_seed, fixed_params, tune_kw, outer_folds=5, log=print):
    """Outer folds measure performance; all tuning happens inside each outer training fold."""
    y = np.asarray(y); P_t = np.zeros(len(y)); P_f = np.zeros(len(y)); chosen = []
    skf = StratifiedKFold(outer_folds, shuffle=True, random_state=outer_seed)
    for k, (tr, te) in enumerate(skf.split(X, y)):
        s = outer_seed * 100 + k
        if kind == "gbm":
            best, _ = tune_lgbm(X.iloc[tr], y[tr], s, **tune_kw)
            m_t, m_f = lgbm(best, s), lgbm(fixed_params, s)
            chosen.append({k2: best[k2] for k2 in ("num_leaves", "learning_rate", "n_estimators", "min_child_samples",
                                                   "subsample", "colsample_bytree", "reg_lambda") if k2 in best})
        else:
            C, _ = tune_logit(X.iloc[tr], y[tr], s, max_rows=tune_kw.get("max_rows", 50000))
            m_t, m_f = logit(C), logit(1.0); chosen.append({"C": C})
        P_t[te] = m_t.fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        P_f[te] = m_f.fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        log(f"    fold {k + 1}/{outer_folds}: tuned AUC {roc_auc_score(y[te], P_t[te]):.3f}, "
            f"fixed AUC {roc_auc_score(y[te], P_f[te]):.3f}, chosen {chosen[-1]}")
    return P_t, P_f, chosen


# ---------- B. overfitting and out-of-sample stability (Berka) ----------
def cv_auc(X, y, make, seed, folds=5):
    y = np.asarray(y); P = np.zeros(len(y))
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y):
        P[te] = make(seed).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y, P), P


def repeat_spread(X, y, make, seeds):
    a = [cv_auc(X, y, make, s)[0] for s in seeds]
    return {"auc_mean": float(np.mean(a)), "auc_sd": float(np.std(a)), "auc_min": float(np.min(a)),
            "auc_max": float(np.max(a)), "n_repeats": len(a)}


def train_test_gap(X, y, make, seeds, folds=5):
    """Mean AUC on the training folds vs on the held-out folds."""
    y = np.asarray(y); tr_a, te_a = [], []
    for s in seeds:
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=s).split(X, y):
            m = make(s).fit(X.iloc[tr], y[tr])
            tr_a.append(roc_auc_score(y[tr], m.predict_proba(X.iloc[tr])[:, 1]))
            te_a.append(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]))
    return {"train_auc": float(np.mean(tr_a)), "test_auc": float(np.mean(te_a)), "gap": float(np.mean(tr_a) - np.mean(te_a))}


def permutation_test(X, y, make, n_perm=200, seed=0):
    """Ojala and Garriga (2010): AUC with shuffled labels gives the chance distribution."""
    y = np.asarray(y); obs = cv_auc(X, y, make, seed)[0]; rng = np.random.default_rng(seed); null = []
    for _ in range(n_perm):
        null.append(cv_auc(X, rng.permutation(y), make, seed)[0])
    null = np.array(null)
    return {"observed_auc": float(obs), "null_mean": float(null.mean()), "null_95th": float(np.percentile(null, 95)),
            "null_max": float(null.max()), "p_value": float((1 + (null >= obs).sum()) / (n_perm + 1)), "n_perm": n_perm}


def learning_curve(X, y, make, fractions=(0.2, 0.4, 0.6, 0.8, 1.0), seeds=range(10), folds=5):
    y = np.asarray(y); out = {}
    for f in fractions:
        a = []
        for s in seeds:
            rng = np.random.default_rng(s)
            for tr, te in StratifiedKFold(folds, shuffle=True, random_state=s).split(X, y):
                if f < 1.0:
                    tr, _ = train_test_split(tr, train_size=f, stratify=y[tr], random_state=s)
                if y[tr].min() == y[tr].max():
                    continue
                a.append(roc_auc_score(y[te], make(s).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]))
        out[str(f)] = {"n_train_approx": int(len(y) * (folds - 1) / folds * f), "auc_mean": float(np.mean(a)),
                       "auc_sd": float(np.std(a))}
    return out


def out_of_time(df, cols, y_col, date_col, make, train_share=0.7, seeds=range(10), n_boot=1000):
    """Train on the earliest loans, test on the latest ones."""
    d = df.sort_values(date_col).reset_index(drop=True)
    cut = int(len(d) * train_share); tr, te = d.iloc[:cut], d.iloc[cut:]
    P = np.mean([make(s).fit(tr[cols], tr[y_col].values).predict_proba(te[cols])[:, 1] for s in seeds], axis=0)
    yt = te[y_col].values
    return {"train_period": [str(tr[date_col].min())[:10], str(tr[date_col].max())[:10]],
            "test_period": [str(te[date_col].min())[:10], str(te[date_col].max())[:10]],
            "n_train": int(len(tr)), "n_test": int(len(te)), "defaults_train": int(tr[y_col].sum()),
            "defaults_test": int(yt.sum()), "auc": float(roc_auc_score(yt, P)), "auc_ci95": auc_ci(yt, P, n_boot)}


# ---------- C. class imbalance ----------
def focal_objective(gamma=2.0):
    """Binary focal loss (Lin et al. 2017) for LightGBM; gradients by central differences on the raw margin."""
    def loss(z, y):
        p = 1 / (1 + np.exp(-z)); pt = np.where(y == 1, p, 1 - p)
        return -((1 - pt) ** gamma) * np.log(np.clip(pt, 1e-12, 1))

    def obj(y_true, z):
        h = 1e-3
        l0, lp, lm = loss(z, y_true), loss(z + h, y_true), loss(z - h, y_true)
        grad = (lp - lm) / (2 * h); hess = np.maximum((lp - 2 * l0 + lm) / h ** 2, 1e-6)
        return grad, hess
    return obj


def smote(X, y, ratio=1.0, k=5, seed=0):
    """Chawla et al. (2002): synthetic minority points on lines between a minority point and one of its k neighbours.
    X must have no missing values. ratio = minority / majority after resampling."""
    rng = np.random.default_rng(seed); X = np.asarray(X, float); y = np.asarray(y)
    mino = X[y == 1]; n_new = int(ratio * (y == 0).sum()) - len(mino)
    if n_new <= 0:
        return X, y
    nn = NearestNeighbors(n_neighbors=k + 1).fit(mino); _, idx = nn.kneighbors(mino)
    base = rng.integers(0, len(mino), n_new); nb = idx[base, rng.integers(1, k + 1, n_new)]
    gap = rng.random((n_new, 1)); new = mino[base] + gap * (mino[nb] - mino[base])
    return np.vstack([X, new]), np.concatenate([y, np.ones(n_new, int)])


def fit_predict_imbalance(method, Xtr, ytr, Xte, params, seed):
    ytr = np.asarray(ytr)
    if method == "none":
        return lgbm(params, seed).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    if method == "class_weight":
        spw = (ytr == 0).sum() / (ytr == 1).sum()
        return lgbm(params, seed, scale_pos_weight=spw).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    if method == "focal_loss":
        m = lgbm(params, seed, objective=focal_objective(2.0)).fit(Xtr, ytr)
        z = m.predict(Xte, raw_score=True)
        return 1 / (1 + np.exp(-z))
    if method == "undersample":
        rng = np.random.default_rng(seed); pos = np.where(ytr == 1)[0]; neg = np.where(ytr == 0)[0]
        n = min(len(pos), len(neg))  # keep every default, sample the same number of repaid loans
        keep = np.concatenate([pos, rng.choice(neg, n, replace=False)])
        return lgbm(params, seed).fit(Xtr.iloc[keep], ytr[keep]).predict_proba(Xte)[:, 1]
    if method == "smote":
        imp = SimpleImputer(strategy="median").fit(Xtr)
        Xs, ys = smote(imp.transform(Xtr), ytr, ratio=1.0, seed=seed)
        cols = list(Xtr.columns)
        m = lgbm(params, seed).fit(pd.DataFrame(Xs, columns=cols), ys)
        return m.predict_proba(pd.DataFrame(imp.transform(Xte), columns=cols))[:, 1]
    raise ValueError(method)


def imbalance_cv(X, y, methods, params, seed=11, folds=5, log=print):
    y = np.asarray(y); out = {}
    splits = list(StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y))
    for meth in methods:
        t0 = time.time(); P = np.zeros(len(y))
        for tr, te in splits:
            P[te] = fit_predict_imbalance(meth, X.iloc[tr], y[tr], X.iloc[te], params, seed)
        out[meth] = P
        log(f"  {meth:13s} AUC {roc_auc_score(y, P):.4f}  ({time.time() - t0:.0f}s)")
    return out


# ---------- E. fairness mitigation ----------
def reweighing_weights(y, g):
    """Kamiran and Calders (2012): weight each (group, outcome) cell so outcome and group are independent."""
    d = pd.DataFrame({"y": np.asarray(y), "g": np.asarray(g)}); n = len(d)
    pg = d["g"].map(d["g"].value_counts() / n); py = d["y"].map(d["y"].value_counts() / n)
    pgy = d.groupby(["g", "y"])["y"].transform("size") / n
    return (pg * py / pgy).values


def age_proxies(X, age, threshold=0.2):
    rho = {c: float(stats.spearmanr(X[c], age, nan_policy="omit").statistic) for c in X.columns}
    rho = dict(sorted(rho.items(), key=lambda kv: -abs(kv[1])))
    chosen = [c for c, r in rho.items() if abs(r) >= threshold]
    return (chosen or list(rho)[:3]), rho  # if nothing passes the threshold, drop the three closest to age


def group_thresholds(p_cal, y_cal, g_cal, mode, approve_rate=0.7):
    """Score cut-offs per group, learned on calibration (out-of-fold training) scores.
    mode 'equal_opportunity': same approval rate among good borrowers in every group, overall approval = approve_rate.
    mode 'equal_approval':    same approval rate in every group."""
    p_cal = np.asarray(p_cal); y_cal = np.asarray(y_cal); g_cal = np.asarray(g_cal); groups = pd.unique(g_cal)
    if mode == "equal_approval":
        return {g: float(np.quantile(p_cal[g_cal == g], approve_rate)) for g in groups}

    def thr_for(t):
        return {g: float(np.quantile(p_cal[(g_cal == g) & (y_cal == 0)], t)) for g in groups}

    def overall(t):
        th = thr_for(t); return np.mean(p_cal <= pd.Series(g_cal).map(th).values)
    lo, hi = 0.0, 1.0
    for _ in range(30):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if overall(mid) < approve_rate else (lo, mid)
    return thr_for((lo + hi) / 2)


def approvals_from_thresholds(p, g, thr):
    t = pd.Series(np.asarray(g)).map(thr).fillna(np.inf).values
    return np.asarray(p) <= t


def fairness_table(y, approve, g, young="<30", old="50+"):
    y = np.asarray(y); approve = np.asarray(approve, bool); g = np.asarray(g); rows = {}
    for k in pd.unique(g):
        m = g == k
        rows[str(k)] = {"n": int(m.sum()), "approval_rate": float(approve[m].mean()),
                        "approval_rate_good": float(approve[m & (y == 0)].mean()),
                        "default_rate_approved": float(y[m & approve].mean()) if (m & approve).any() else None}
    return {"by_group": rows, "overall_approval_rate": float(approve.mean()),
            "default_rate_approved": float(y[approve].mean()),
            "approval_ratio_young_vs_old": rows[young]["approval_rate"] / rows[old]["approval_rate"],
            "good_borrower_gap_young_minus_old": rows[young]["approval_rate_good"] - rows[old]["approval_rate_good"]}


def oof_scores(X, y, make, seed, folds=5, weights=None):
    y = np.asarray(y); P = np.zeros(len(y))
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y):
        kw = {} if weights is None else {"sample_weight": weights[tr]}
        P[te] = make(seed).fit(X.iloc[tr], y[tr], **kw).predict_proba(X.iloc[te])[:, 1]
    return P


def done(name):
    return os.path.exists(f'{PARTS}/{name}.json')

def save(name, obj):
    with open(f'{PARTS}/{name}.json', 'w') as f:
        json.dump(obj, f, indent=1, default=str)
    print('Saved section', name)

In [ ]:
# Load data
p = f'{OUT}/home_credit_features.parquet'
hc = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv(f'{OUT}/home_credit_features.csv.gz')
y_hc = hc['TARGET'].values
BUR = [c for c in hc.columns if c.startswith('bur_')]
ALT = [c for c in hc.columns if c.startswith('alt_')]
COMB = BUR + ALT
EXT = ['bur_ext_source_1', 'bur_ext_source_2', 'bur_ext_source_3']
thin = hc['thin_file'].values == 1
GROUPS = {'all': None, 'thin_file': thin, 'with_bureau': ~thin}
tables = {n: pd.read_csv(f'{OUT}/berka_raw/{n}.csv') for n in ['loan', 'account', 'trans', 'disp', 'client']}
bk = build_berka(tables)
bk = bk[bk['enough_history'] == 1].reset_index(drop=True)
bk['loan_date'] = pd.to_datetime(bk['loan_date'])
y_bk = bk['default'].values
CF = [c for c in bk.columns if c.startswith('cf_')]
TERMS = [c for c in bk.columns if c.startswith('terms_')]
print(f'Home Credit {len(hc):,} applicants; Berka {len(bk)} loans, {y_bk.sum()} defaults, {len(CF)} cash-flow features')

In [ ]:
# A. Home Credit: nested cross-validation with hyperparameter search (about 25-30 minutes)
if not done('A_hc_tuning'):
    t0 = time.time(); res = {'settings': {'outer_folds': 5, 'outer_seed': 11, 'inner_folds': 3, 'random_search_candidates': 12,
                                          'plus_hand_set_configuration': True, 'inner_rows': 50000,
                                          'early_stopping_rounds': 50, 'logit_C_grid': LOGIT_C, 'space': HC_SPACE}}
    P = {}
    for kind in ['gbm', 'logit']:
        for name, cols in [('bureau', BUR), ('combined', COMB)]:
            print(kind, name)
            Pt, Pf, chosen = nested_cv(hc[cols], y_hc, kind, 11, HC_FIXED,
                                       dict(space=HC_SPACE, n_iter=12, max_rows=50000, early_stop=True))
            P[(kind, name)] = (Pt, Pf)
            res[f'{kind}|{name}'] = {'chosen_per_fold': chosen,
                                      'tuned': {g: summarise(y_hc, Pt, m, n_boot=100) for g, m in GROUPS.items()},
                                      'hand_set': {g: summarise(y_hc, Pf, m, n_boot=100) for g, m in GROUPS.items()}}
    res['delong_combined_vs_bureau'] = {}
    for kind in ['gbm', 'logit']:
        for which, j in [('tuned', 0), ('hand_set', 1)]:
            for g, m in GROUPS.items():
                mm = np.ones(len(y_hc), bool) if m is None else m
                res['delong_combined_vs_bureau'][f'{kind}|{which}|{g}'] = delong_test(
                    y_hc[mm], P[(kind, 'combined')][j][mm], P[(kind, 'bureau')][j][mm])
    res['minutes'] = (time.time() - t0) / 60
    save('A_hc_tuning', res)
    for k, v in res.items():
        if '|' in k and 'tuned' in v:
            print(k, {g: (round(v['hand_set'][g]['auc'], 4), round(v['tuned'][g]['auc'], 4)) for g in GROUPS})

In [ ]:
# B. Berka: nested tuning and overfitting checks (about 15-20 minutes)
if not done('B_berka'):
    t0 = time.time(); res = {}
    gbm_fixed = lambda s: lgbm(BK_FIXED, s)
    X = bk[CF]
    # B1 nested tuning: full grid inside each training fold, 10 repeats of 5-fold
    for kind in ['gbm', 'logit']:
        Pts, Pfs, ch = [], [], []
        for r in range(10):
            Pt, Pf, chosen = nested_cv(X, y_bk, kind, r, BK_FIXED,
                                       dict(space=BK_GRID, early_stop=False, max_rows=None), log=lambda *a: None)
            Pts.append(roc_auc_score(y_bk, Pt)); Pfs.append(roc_auc_score(y_bk, Pf)); ch += chosen
        res[f'nested_{kind}'] = {'tuned_auc_mean': float(np.mean(Pts)), 'tuned_auc_sd': float(np.std(Pts)),
                                 'hand_set_auc_mean': float(np.mean(Pfs)), 'hand_set_auc_sd': float(np.std(Pfs)),
                                 'chosen_counts': pd.Series([json.dumps(c, sort_keys=True) for c in ch]).value_counts().head(5).to_dict()}
        print(kind, res[f'nested_{kind}'])
    # B2 spread over 20 repeats, B3 train vs test gap
    res['repeat_spread'] = {'gbm': repeat_spread(X, y_bk, gbm_fixed, range(20)),
                            'logit': repeat_spread(X, y_bk, lambda s: logit(1.0), range(20))}
    res['train_test_gap'] = {'gbm': train_test_gap(X, y_bk, gbm_fixed, range(10)),
                             'logit': train_test_gap(X, y_bk, lambda s: logit(1.0), range(10))}
    print('spread', res['repeat_spread']); print('gap', res['train_test_gap'])
    # B4 permutation test (labels shuffled 200 times)
    res['permutation_test_gbm'] = permutation_test(X, y_bk, gbm_fixed, n_perm=200)
    print('permutation', res['permutation_test_gbm'])
    # B5 learning curve
    res['learning_curve_gbm'] = learning_curve(X, y_bk, gbm_fixed)
    print('learning curve', {k: round(v['auc_mean'], 3) for k, v in res['learning_curve_gbm'].items()})
    # B6 out-of-time: train on the earliest 70% of loans, test on the latest 30%
    res['out_of_time'] = {'gbm_cashflow': out_of_time(bk, CF, 'default', 'loan_date', gbm_fixed),
                          'logit_cashflow': out_of_time(bk, CF, 'default', 'loan_date', lambda s: logit(1.0)),
                          'gbm_loan_terms': out_of_time(bk, TERMS, 'default', 'loan_date', gbm_fixed)}
    print('out of time', {k: (round(v['auc'], 3), v['defaults_test']) for k, v in res['out_of_time'].items()})
    # B7 shorter transaction windows (same loans)
    res['window_length'] = {}
    for w in [90, 180, 365]:
        bw = build_berka(tables, window_days=w)
        bw = bw[bw['loan_id'].isin(bk['loan_id'])].reset_index(drop=True)
        yw = bw['default'].values
        res['window_length'][f'{w}_days'] = {k: repeat_spread(bw[CF], yw, mk, range(10))
                                             for k, mk in [('gbm', gbm_fixed), ('logit', lambda s: logit(1.0))]}
        print(w, {k: round(v['auc_mean'], 3) for k, v in res['window_length'][f'{w}_days'].items()})
    # B8 class imbalance on Berka
    res['imbalance'] = {}
    for meth in ['none', 'class_weight', 'smote']:
        a = []
        for s in range(10):
            Pm = imbalance_cv(X, y_bk, [meth], BK_FIXED, seed=s, log=lambda *x: None)[meth]
            a.append(summarise(y_bk, Pm, n_boot=50))
        res['imbalance'][meth] = {k: float(np.mean([r[k] for r in a])) for k in
                                  ['auc', 'pr_auc', 'brier', 'ece', 'default_rate_approved', 'share_of_defaulters_declined']}
        print(meth, {k: round(v, 3) for k, v in res['imbalance'][meth].items()})
    res['minutes'] = (time.time() - t0) / 60
    save('B_berka', res)

In [ ]:
# C. Home Credit: class imbalance (about 20 minutes)
if not done('C_hc_imbalance'):
    t0 = time.time()
    methods = ['none', 'class_weight', 'focal_loss', 'undersample', 'smote']
    P = imbalance_cv(hc[COMB], y_hc, methods, HC_FIXED, seed=11)
    res = {m: {g: summarise(y_hc, P[m], mk, n_boot=100) for g, mk in GROUPS.items() if g != 'with_bureau'} for m in methods}
    res['delong_vs_none'] = {m: delong_test(y_hc, P[m], P['none']) for m in methods if m != 'none'}
    res['minutes'] = (time.time() - t0) / 60
    save('C_hc_imbalance', res)
    print(pd.DataFrame({m: {k: round(v, 4) for k, v in res[m]['all'].items() if k not in ('auc_ci95', 'n')} for m in methods}))

In [ ]:
# D. Home Credit: dependence on the undocumented external scores (about 20 minutes)
if not done('D_ext_scores'):
    t0 = time.time()
    no2 = lambda cols: [c for c in cols if c != 'bur_ext_source_2']
    noext = lambda cols: [c for c in cols if c not in EXT]
    sets = {'bureau': BUR, 'bureau_without_ext2': no2(BUR), 'bureau_without_all_ext': noext(BUR),
            'combined': COMB, 'combined_without_ext2': no2(COMB), 'combined_without_all_ext': noext(COMB)}
    P = {}; res = {'feature_counts': {k: len(v) for k, v in sets.items()}}
    for name, cols in sets.items():
        P[name] = oof_predictions(hc[cols], y_hc, 'gbm', [11])[:, 0]
        res[name] = {g: summarise(y_hc, P[name], m, n_boot=100) for g, m in GROUPS.items()}
        print(f"{name:26s}", {g: round(res[name][g]['auc'], 4) for g in GROUPS})
    res['delong'] = {}
    for a, b in [('combined_without_ext2', 'bureau_without_ext2'), ('combined_without_all_ext', 'bureau_without_all_ext'),
                 ('combined_without_ext2', 'combined'), ('combined_without_all_ext', 'combined')]:
        for g, m in GROUPS.items():
            mm = np.ones(len(y_hc), bool) if m is None else m
            res['delong'][f'{a} vs {b}|{g}'] = delong_test(y_hc[mm], P[a][mm], P[b][mm])
    res['ext2_missing_share'] = {g: float(hc.loc[(np.ones(len(hc), bool) if m is None else m), 'bur_ext_source_2'].isna().mean())
                                 for g, m in GROUPS.items()}
    res['minutes'] = (time.time() - t0) / 60
    save('D_ext_scores', res)

In [ ]:
# E. Fairness mitigation by age band (about 10 minutes). Same 80/20 split as Notebooks 3 and 4.
if not done('E_fairness_mitigation'):
    from sklearn.model_selection import train_test_split
    t0 = time.time()
    idx_tr, idx_te = train_test_split(np.arange(len(hc)), test_size=0.2, stratify=y_hc, random_state=0)
    tr, te = hc.iloc[idx_tr].reset_index(drop=True), hc.iloc[idx_te].reset_index(drop=True)
    ytr, yte = y_hc[idx_tr], y_hc[idx_te]
    gtr, gte = tr['audit_age_band'].values, te['audit_age_band'].values
    thin_tr, thin_te = tr['thin_file'].values == 1, te['thin_file'].values == 1
    mk = lambda s: lgbm(HC_FIXED, s)
    scores, uses_age = {}, {}

    # E0 baseline
    base = mk(11).fit(tr[COMB], ytr); scores['baseline'] = base.predict_proba(te[COMB])[:, 1]; uses_age['baseline'] = 'no'
    # E1 reweighing (age used only to weight training data)
    w = reweighing_weights(ytr, gtr)
    scores['reweighing'] = mk(11).fit(tr[COMB], ytr, sample_weight=w).predict_proba(te[COMB])[:, 1]
    uses_age['reweighing'] = 'training only'
    # E2 remove features that track age (|Spearman rho| >= 0.2 on training data)
    proxies, rho = age_proxies(tr[COMB], tr['audit_age'].values, 0.2)
    keep = [c for c in COMB if c not in proxies]
    scores['remove_age_proxies'] = mk(11).fit(tr[keep], ytr).predict_proba(te[keep])[:, 1]
    uses_age['remove_age_proxies'] = 'no'
    print('Age proxies removed:', proxies)

    # E3/E4 group thresholds learned on out-of-fold training scores (no test labels used)
    p_cal = oof_scores(tr[COMB], ytr, mk, 11)
    res = {'age_proxies_removed': proxies, 'spearman_with_age_top10': dict(list(rho.items())[:10]), 'methods': {}}
    for scope, mtr, mte in [('all', np.ones(len(tr), bool), np.ones(len(te), bool)), ('thin_file', thin_tr, thin_te)]:
        for name, p in scores.items():
            cut = np.quantile(p[mte], 0.7)
            ap = p[mte] <= cut
            res['methods'].setdefault(name, {'uses_age_at_decision': 'no', 'uses_age': uses_age[name]})[scope] = {
                'auc': float(roc_auc_score(yte[mte], p[mte])), **fairness_table(yte[mte], ap, gte[mte])}
        for mode in ['equal_opportunity', 'equal_approval']:
            thr = group_thresholds(p_cal[mtr], ytr[mtr], gtr[mtr], mode, 0.7)
            ap = approvals_from_thresholds(scores['baseline'][mte], gte[mte], thr)
            name = f'thresholds_{mode}'
            res['methods'].setdefault(name, {'uses_age_at_decision': 'yes', 'uses_age': 'decision time'})[scope] = {
                'auc': float(roc_auc_score(yte[mte], scores['baseline'][mte])), 'thresholds': thr,
                **fairness_table(yte[mte], ap, gte[mte])}
    res['minutes'] = (time.time() - t0) / 60
    save('E_fairness_mitigation', res)
    for name, v in res['methods'].items():
        for scope in ['all', 'thin_file']:
            r = v[scope]
            print(f"{name:32s} {scope:9s} AUC {r['auc']:.3f} approve {r['overall_approval_rate']:.3f} "
                  f"default|approved {r['default_rate_approved']:.4f} ratio<30/50+ {r['approval_ratio_young_vs_old']:.2f} "
                  f"good-gap {r['good_borrower_gap_young_minus_old']:+.3f}")

In [ ]:
# Combine all sections and download
allres = {}
for n in ['A_hc_tuning', 'B_berka', 'C_hc_imbalance', 'D_ext_scores', 'E_fairness_mitigation']:
    if done(n):
        allres[n] = json.load(open(f'{PARTS}/{n}.json'))
    else:
        print('Missing section:', n, '- run its cell again')
with open(f'{OUT}/robustness_results.json', 'w') as f:
    json.dump(allres, f, indent=1, default=str)
from google.colab import files
files.download(f'{OUT}/robustness_results.json')
print('Saved. Send robustness_results.json to Claude.')